# **PROJET DE MODELES DE COURBE DE TAUX**
* Mariane ALAPINI & Céleste NENEHIDINI*

In [ ]:
# Bibliothèques à importer
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## **1. Reconstitution d'une courbe de taux zéro coupon**

### 1.1. Formule de valorisation des taux de marché

**Question 1** : 

Le tableau 1 décrit une courbe de taux interbancaires. Les taux observés proviennent de trois segments à savoir : MM (Money Market), FUT (futures de taux), et SWAP. 

**Question 2** : 

La courbe interbancaire est une courbe de taux qui représente les taux d'intérêt auxquels les banques se prêtent de l'argent entre elles. Elle est utilisée pour déterminer les taux d'intérêt des prêts et des emprunts à court terme. Elle est construite sur le court terme (maturité<6M) à partir des taux du marchés monétaire (Money Market) basés sur les dépots non garantis entre banques. Sur le moyen terme (6m - 3y) elle est construite à partir des contrats futures, i.e. des forwards sur un marché OTC (Over The Counter) et sur le long terme (>3y) elle est construite à partir des contrats de swap euribor (Euro Interbank Offered Rate) 3M ou 6M.

En l'absence d'oppotunité d'arbitrage, les valorisations des instruments de marché s'expriment en fonction des taux zéro coupon implicites ci-dessous :
- Sur le segment **Money Market**, on cote en taux monétaires :

    $$
    L_t(T,T+\delta) = \frac{1}{\delta} \left( \frac{B(t,T)}{B(t,T+\delta)} - 1 \right),
    $$

    avec t le temps courant, T la maturité et $\delta$ la période de capitalisation.
    Dans notre cas, t=T=0 et $\delta$ varie en fonction de la maturité.

- Sur le segment **Future**, on côte en 1 - tx forward :

    $$
    future(T, T+\delta) = 1 - L_t(T,T+\delta).
    $$
    Dans notre cas, t=0, T= maturité - 3m et $\delta = 3m$.


- Sur le segment **Swap**, on côte en taux swap :

    $$
    Swap(t, T_0, T_n) = B(t, T_0)−B(t, T_n)−K ×lvl(t),
    $$

    avec K le taux fixe du swap qui égalise la PV du swap vaut 0 et $lvl(t)=\sum_{i=1}^{n} \delta_i B(t, T_i)$ le taux de marché à la maturité $T_n$.
    Dans notre cas, la date de départ est le spot, i.e. $T_0=0$ et $T_n$ est la maturité du swap, et t=0 (vu d'aujourd'hui).

**Question 3** : Le principe du bootstrapping

Le bootstrapping est une méthode qui permet de reconstituer la courbe des taux zéro-coupon de manière progressive. Il consiste à extraire les taux zéro-coupon maturité par maturité, en commençant par les instruments de marché les plus courts. Les taux ainsi obtenus sont ensuite réutilisés pour valoriser les instruments de maturité plus longue, ce qui permet d’identifier successivement l’ensemble de la courbe zéro-coupon. En pratique, cette construction s’effectue généralement dans l’ordre suivant : marché monétaire (MM), contrats futures (FUT), puis swaps de taux (SWAP).

Cette étape est indispensable car les taux zéro-coupon ne sont pas directement observables sur les marchés financiers. Or, ils constituent un élément central de la modélisation en taux d’intérêt : ils sont nécessaires pour valoriser les produits de taux, calculer les taux forward implicites et actualiser les flux de trésorerie futurs. En l’absence de bootstrapping, il ne serait pas possible de garantir la cohérence entre les différents instruments de marché ni d’assurer l’absence d’opportunités d’arbitrage. Le bootstrapping représente ainsi une étape fondamentale de toute valorisation en taux d’intérêt.

**Question 4** : Différence entre forward et future

Un contrat forward est un instrument de gré à gré, négocié directement entre deux contreparties. Son règlement intervient uniquement à l’échéance du contrat et il ne donne pas lieu à des appels de marge intermédiaires. La valeur du contrat est donc réalisée en une seule fois, à maturité.

À l’inverse, un contrat future est un instrument standardisé, négocié sur un marché organisé. Sa principale spécificité réside dans le mécanisme de règlement quotidien par mark-to-market : les gains et les pertes sont constatés et réglés chaque jour, ce qui entraîne des appels de marge journaliers.

Bien que forwards et futures portent sur des sous-jacents similaires, leurs prix ne sont pas nécessairement identiques. Cet écart s’explique par la corrélation entre l’évolution des taux d’intérêt et les flux de trésorerie générés par le contrat. En présence de volatilité des taux, les gains ou pertes d’un future sont réalisés avant l’échéance, ce qui affecte sa valeur économique. À l’inverse, dans un contrat forward, le règlement intervient uniquement à maturité, sans flux intermédiaires. Cette différence de calendrier des flux explique l’écart potentiel entre le prix d’un forward et celui d’un future.


### 1.2. Construction de la courbe des taux zéro-coupon


In [ ]:
# Importation du jeu de données
data = pd.read_csv("Data(ZC).csv", header=None)

data.columns = [
    "Type",   # MM / FUT / SWAP
    "T",     # en années
    "Market_Quote", # Cot SHIFT
    "ZC_Rate",      # ZC SHIFT
    "Discount_Factor"  # B(0,T)
]

data.head()